# UVA Medical Center Quality & Performance Benchmarking

## Introduction

This project evaluates UVA Medical Center across selected hospital quality measures using publicly available CMS hospital data. The analysis compares UVA with other Virginia hospitals across four areas: timely care, unplanned visits/readmissions, healthcare-associated infections, and mortality/complications.

Five CMS datasets are cleaned, standardized, and connected through the CMS Facility ID. UVA Medical Center is treated as the focal hospital, while other Virginia hospitals form the peer comparison population. UVA is excluded from its own peer benchmark calculations so its score does not influence the baseline used for comparison.

Since the measures use different units and scales, I used z-scores and performance direction to make the comparisons easier to interpret.

## Import Libraries

The core Python libraries used for data loading, cleaning, transformation, and analysis are imported below.

In [5]:
import pandas as pd
import numpy as np




---


## Load CMS Hospital Datasets

Five public CMS hospital datasets are loaded to analyze UVA Medical Center across hospital characteristics, timely care, readmissions, infections, and clinical outcomes.

In [6]:
from google.colab import files
uploaded = files.upload()


Saving Timely_Effective_Care.csv to Timely_Effective_Care.csv


In [7]:
hospital_info = pd.read_csv(
    "Hospital_General_Info.csv", dtype={"Facility ID": "string"}, low_memory=False
)
timely_care = pd.read_csv(
    "Timely_Effective_Care.csv", dtype={"Facility ID": "string"}, low_memory=False
)
unplanned_visits = pd.read_csv(
    "Unplanned_Hospital_Visits.csv", dtype={"Facility ID": "string"}, low_memory=False
)
infections = pd.read_csv(
    "Hospital_Infections-Hospital.csv", dtype={"Facility ID": "string"}, low_memory=False
)
complications = pd.read_csv(
    "Complications_mortality.csv", dtype={"Facility ID": "string"}, low_memory=False
)

In [8]:
datasets = {
    "Hospital Information": hospital_info,
    "Timely_Effective_Care.csv": timely_care,
    "Unplanned Visits": unplanned_visits,
    "Infections": infections,
    "Complications & Mortality": complications
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Hospital Information: (5419, 38)
Timely_Effective_Care.csv: (138084, 16)
Unplanned Visits: (67060, 20)
Infections: (172440, 15)
Complications & Mortality: (95800, 18)




---


## Dataset Overview

The dimensions and column structures of each dataset are reviewed to understand how the files are organized before cleaning.

In [9]:
for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print(df.columns.tolist())


Hospital Information
['Facility ID', 'Facility Name', 'Address', 'City/Town', 'State', 'ZIP Code', 'County/Parish', 'Telephone Number', 'Hospital Type', 'Hospital Ownership', 'Emergency Services', 'Meets criteria for birthing friendly designation', 'Hospital overall rating', 'Hospital overall rating footnote', 'MORT Group Measure Count', 'Count of Facility MORT Measures', 'Count of MORT Measures Better', 'Count of MORT Measures No Different', 'Count of MORT Measures Worse', 'MORT Group Footnote', 'Safety Group Measure Count', 'Count of Facility Safety Measures', 'Count of Safety Measures Better', 'Count of Safety Measures No Different', 'Count of Safety Measures Worse', 'Safety Group Footnote', 'READM Group Measure Count', 'Count of Facility READM Measures', 'Count of READM Measures Better', 'Count of READM Measures No Different', 'Count of READM Measures Worse', 'READM Group Footnote', 'Pt Exp Group Measure Count', 'Count of Facility Pt Exp Measures', 'Pt Exp Group Footnote', 'TE Gro

## Standardize Facility IDs

CMS Facility IDs are standardized across datasets so hospitals can be matched reliably between tables. IDs are stored as six-character strings to preserve leading zeros.

In [10]:
# Preserve CMS Facility IDs as six-character text values.
for name, df in datasets.items():
    if "Facility ID" in df.columns:
        df["Facility ID"] = (
            df["Facility ID"]
            .astype("string")
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(6)
        )

## Identify UVA Medical Center

UVA Medical Center is identified using its CMS Facility ID so its performance can later be compared with other Virginia hospitals.

In [11]:
hospital_info[hospital_info["Facility Name"].str.contains("University OF Virginia", case=False, na=False)]

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
4910,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,Acute Care Hospitals,Government - State,...,2,8,1,NaN,15,15,NaN,10,8,NaN


In [12]:
uva_id = "490009"

In [13]:
for name, df in datasets.items():
  if "Facility ID" in df.columns:
    count = (df["Facility ID"] == uva_id).sum()
    print(f"{name}:{count}")

Hospital Information:1
Timely_Effective_Care.csv:30
Unplanned Visits:14
Infections:36
Complications & Mortality:20


## Create Virginia Hospital Comparison Group

Virginia hospitals are isolated to create a relevant benchmark group for evaluating UVA Medical Center.

In [14]:
virginia_hospitals = hospital_info[hospital_info["State"] == "VA"].copy()
print("virginia_hospitals:", len(virginia_hospitals))

virginia_hospitals: 96


In [15]:
virginia_hospitals[["Facility ID", "Facility Name", "City/Town", "Hospital Type"]].head()

,Facility ID,Facility Name,City/Town,Hospital Type
4906,490002,RUSSELL COUNTY HOSPITAL,LEBANON,Acute Care Hospitals
4907,490004,SENTARA RMH MEDICAL CENTER,HARRISONBURG,Acute Care Hospitals
4908,490005,WINCHESTER MEDICAL CENTER,WINCHESTER,Acute Care Hospitals
4909,490007,SENTARA NORFOLK GENERAL HOSPITAL,NORFOLK,Acute Care Hospitals
4910,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,CHARLOTTESVILLE,Acute Care Hospitals


In [16]:
for name, df in datasets.items():
    print("\n", name)
    print(df.isnull().sum().sort_values(ascending=False).head(10))


 Hospital Information
TE Group Footnote                                   4411
READM Group Footnote                                4196
MORT Group Footnote                                 4023
Pt Exp Group Footnote                               3403
Safety Group Footnote                               3270
Meets criteria for birthing friendly designation    3156
Hospital overall rating footnote                    3103
State                                                  0
ZIP Code                                               0
Address                                                0
dtype: int64

 Timely_Effective_Care.csv
Footnote            46249
Sample               4658
Facility ID             0
Facility Name           0
State                   0
ZIP Code                0
Address                 0
City/Town               0
Telephone Number        0
County/Parish           0
dtype: int64

 Unplanned Visits
Footnote         31264
Facility ID          0
Address              0
Facil

In [17]:
for name, df in datasets.items():
  print("\n", name)
  print(df.dtypes)


 Hospital Information
Facility ID                                         string[python]
Facility Name                                               object
Address                                                     object
City/Town                                                   object
State                                                       object
ZIP Code                                                     int64
County/Parish                                               object
Telephone Number                                            object
Hospital Type                                               object
Hospital Ownership                                          object
Emergency Services                                          object
Meets criteria for birthing friendly designation            object
Hospital overall rating                                     object
Hospital overall rating footnote                            object
MORT Group Measure Count               

## Create Working Copies

Copies of the original datasets are created so cleaning operations can be performed without modifying the raw imported data.

In [18]:
hospital_clean = hospital_info.copy()
timely_clean = timely_care.copy()
unplanned_clean = unplanned_visits.copy()
infections_clean = infections.copy()
complications_clean = complications.copy()

In [19]:
clean_datasets = {
    "Hospital Information": hospital_clean,
    "Timely & Effective Care": timely_clean,
    "Unplanned Visits": unplanned_clean,
    "Infections": infections_clean,
    "Complications & Mortality": complications_clean
}

for name, df in clean_datasets.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    display(df.head())


Hospital Information


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,1,9,1,NaN,15,15,NaN,10,10,NaN
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,1,8,0,NaN,15,15,NaN,10,10,NaN
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,1,8,0,NaN,15,15,NaN,10,9,NaN
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,0,3,2,NaN,15,5,NaN,10,7,NaN
4,010011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,1,5,2,29.0,15,10,29.0,10,7,29.0



Timely & Effective Care


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Condition,Measure ID,Measure Name,Score,Sample,Footnote,Start Date,End Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Emergency Department,EDV,Emergency department volume,very high,NaN,NaN,01/01/2024,12/31/2024
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS,Global Malnutrition Composite Score,Not Available,Not Available,5,01/01/2024,12/31/2024
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS_Malnutrition_Diagnosis_Documented,Global Malnutrition Composite Score: Malnutrit...,Not Available,Not Available,5,01/01/2024,12/31/2024
3,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS_Malnutrition_Screening,Global Malnutrition Composite Score: Malnutrit...,Not Available,Not Available,5,01/01/2024,12/31/2024
4,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS_Nutrition_Assessment,Global Malnutrition Composite Score: Nutrition...,Not Available,Not Available,5,01/01/2024,12/31/2024



Unplanned Visits


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Measure ID,Measure Name,Compared to National,Denominator,Score,Lower Estimate,Higher Estimate,Number of Patients,Number of Patients Returned,Footnote,Start Date,End Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,EDAC_30_AMI,Hospital return days for heart attack patients,Average Days per 100 Discharges,242,-2.6,-47.1,74.7,235,51,NaN,07/01/2022,06/30/2025
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,EDAC_30_HF,Hospital return days for heart failure patients,Average Days per 100 Discharges,659,-14.4,-59.7,52.9,540,173,NaN,07/01/2022,06/30/2025
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,EDAC_30_PN,Hospital return days for pneumonia patients,Average Days per 100 Discharges,586,0.7,-35.1,50,530,126,NaN,07/01/2022,06/30/2025
3,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Hybrid_HWR,Hybrid Hospital-Wide All-Cause Readmission Mea...,Not Available,Not Available,Not Available,Not Available,Not Available,Not Applicable,Not Applicable,4,07/01/2024,06/30/2025
4,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,OP_32,Rate of unplanned hospital visits after colono...,No Different Than the National Rate,234,12.7,9.5,17,Not Applicable,Not Applicable,NaN,01/01/2022,12/31/2024



Infections


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Measure ID,Measure Name,Compared to National,Score,Footnote,Start Date,End Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,HAI_1_CILOWER,Central Line Associated Bloodstream Infection ...,Better than the National Benchmark,0.071,NaN,10/01/2024,09/30/2025
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,HAI_1_CIUPPER,Central Line Associated Bloodstream Infection ...,Better than the National Benchmark,0.761,NaN,10/01/2024,09/30/2025
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,HAI_1_DOPC,Central Line Associated Bloodstream Infection:...,Better than the National Benchmark,10192,NaN,10/01/2024,09/30/2025
3,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,HAI_1_ELIGCASES,Central Line Associated Bloodstream Infection ...,Better than the National Benchmark,10.733,NaN,10/01/2024,09/30/2025
4,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,HAI_1_NUMERATOR,Central Line Associated Bloodstream Infection ...,Better than the National Benchmark,3,NaN,10/01/2024,09/30/2025



Complications & Mortality


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Measure ID,Measure Name,Compared to National,Denominator,Score,Lower Estimate,Higher Estimate,Footnote,Start Date,End Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,COMP_HIP_KNEE,Rate of complications for hip/knee replacement...,No Different Than the National Rate,33,3.8,2.4,6.0,NaN,04/01/2023,03/31/2025
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Hybrid_HWM,Hybrid Hospital-Wide All-Cause Risk Standardiz...,No Different Than the National Rate,4972,4.4,3.9,5.0,NaN,07/01/2024,06/30/2025
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,MORT_30_AMI,Death rate for heart attack patients,No Different Than the National Rate,234,13.5,10.9,16.6,NaN,07/01/2022,06/30/2025
3,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,MORT_30_CABG,Death rate for CABG surgery patients,No Different Than the National Rate,127,3.2,1.7,6.3,NaN,07/01/2022,06/30/2025
4,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,MORT_30_COPD,Death rate for COPD patients,No Different Than the National Rate,116,9.1,6.2,13.2,NaN,07/01/2022,06/30/2025


In [20]:
for name, df in clean_datasets.items():
    print(
        f"{name}: {df.duplicated().sum()} exact duplicate rows"
    )

Hospital Information: 0 exact duplicate rows
Timely & Effective Care: 0 exact duplicate rows
Unplanned Visits: 0 exact duplicate rows
Infections: 0 exact duplicate rows
Complications & Mortality: 0 exact duplicate rows


In [21]:
for name, df in clean_datasets.items():
    if "Facility ID" in df.columns:
        print(
            f"{name}: {df['Facility ID'].nunique()} unique hospitals"
        )

Hospital Information: 5419 unique hospitals
Timely & Effective Care: 4658 unique hospitals
Unplanned Visits: 4790 unique hospitals
Infections: 4790 unique hospitals
Complications & Mortality: 4790 unique hospitals


In [22]:
for name, df in clean_datasets.items():
    if "Facility ID" in df.columns:
        print("\n", name)

        display(
            df[df["Facility ID"] == uva_id].head()
        )


 Hospital Information


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
4910,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,Acute Care Hospitals,Government - State,...,2,8,1,NaN,15,15,NaN,10,8,NaN



 Timely & Effective Care


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Condition,Measure ID,Measure Name,Score,Sample,Footnote,Start Date,End Date
124543,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,Emergency Department,EDV,Emergency department volume,very high,NaN,NaN,01/01/2024,12/31/2024
124544,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,Electronic Clinical Quality Measure,GMCS,Global Malnutrition Composite Score,Not Available,Not Available,4,01/01/2024,12/31/2024
124545,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,Electronic Clinical Quality Measure,GMCS_Malnutrition_Diagnosis_Documented,Global Malnutrition Composite Score: Malnutrit...,Not Available,Not Available,4,01/01/2024,12/31/2024
124546,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,Electronic Clinical Quality Measure,GMCS_Malnutrition_Screening,Global Malnutrition Composite Score: Malnutrit...,Not Available,Not Available,4,01/01/2024,12/31/2024
124547,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,Electronic Clinical Quality Measure,GMCS_Nutrition_Assessment,Global Malnutrition Composite Score: Nutrition...,Not Available,Not Available,4,01/01/2024,12/31/2024



 Unplanned Visits


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Measure ID,Measure Name,Compared to National,Denominator,Score,Lower Estimate,Higher Estimate,Number of Patients,Number of Patients Returned,Footnote,Start Date,End Date
60452,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,EDAC_30_AMI,Hospital return days for heart attack patients,Average Days per 100 Discharges,363,35.9,-13.6,98.8,353,94,NaN,07/01/2022,06/30/2025
60453,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,EDAC_30_HF,Hospital return days for heart failure patients,Average Days per 100 Discharges,763,-8,-45.7,40.3,627,231,NaN,07/01/2022,06/30/2025
60454,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,EDAC_30_PN,Hospital return days for pneumonia patients,Average Days per 100 Discharges,474,-6.7,-34.1,30,436,146,NaN,07/01/2022,06/30/2025
60455,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,Hybrid_HWR,Hybrid Hospital-Wide All-Cause Readmission Mea...,Not Available,Not Available,Not Available,Not Available,Not Available,Not Applicable,Not Applicable,4,07/01/2024,06/30/2025
60456,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,OP_32,Rate of unplanned hospital visits after colono...,Better Than the National Rate,4614,10,8.1,12.3,Not Applicable,Not Applicable,NaN,01/01/2022,12/31/2024



 Infections


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Measure ID,Measure Name,Compared to National,Score,Footnote,Start Date,End Date
155448,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,HAI_1_CILOWER,Central Line Associated Bloodstream Infection ...,Better than the National Benchmark,0.209,NaN,10/01/2024,09/30/2025
155449,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,HAI_1_CIUPPER,Central Line Associated Bloodstream Infection ...,Better than the National Benchmark,0.625,NaN,10/01/2024,09/30/2025
155450,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,HAI_1_DOPC,Central Line Associated Bloodstream Infection:...,Better than the National Benchmark,29956,NaN,10/01/2024,09/30/2025
155451,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,HAI_1_ELIGCASES,Central Line Associated Bloodstream Infection ...,Better than the National Benchmark,34.677,NaN,10/01/2024,09/30/2025
155452,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,HAI_1_NUMERATOR,Central Line Associated Bloodstream Infection ...,Better than the National Benchmark,13,NaN,10/01/2024,09/30/2025



 Complications & Mortality


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Measure ID,Measure Name,Compared to National,Denominator,Score,Lower Estimate,Higher Estimate,Footnote,Start Date,End Date
86360,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,COMP_HIP_KNEE,Rate of complications for hip/knee replacement...,No Different Than the National Rate,131,4.3,2.3,7.5,NaN,04/01/2023,03/31/2025
86361,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,Hybrid_HWM,Hybrid Hospital-Wide All-Cause Risk Standardiz...,No Different Than the National Rate,5417,3.3,3.2,4.0,NaN,07/01/2024,06/30/2025
86362,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,MORT_30_AMI,Death rate for heart attack patients,No Different Than the National Rate,265,10.4,7.8,13.4,NaN,07/01/2022,06/30/2025
86363,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,MORT_30_CABG,Death rate for CABG surgery patients,No Different Than the National Rate,220,2,0.9,4.3,NaN,07/01/2022,06/30/2025
86364,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,1215 LEE STREET,CHARLOTTESVILLE,VA,22908,CHARLOTTESVILLE CITY,(434) 924-0000,MORT_30_COPD,Death rate for COPD patients,No Different Than the National Rate,111,9.5,6.3,14.2,NaN,07/01/2022,06/30/2025


---

# Day 2: Data Cleaning

The CMS datasets are cleaned and standardized for analysis. This process includes handling missing-value labels, standardizing dates and numeric measures, removing exact duplicate records, and preparing Virginia-only analytical tables.

In [23]:
# Convert common CMS unavailable-value labels to standard missing values.
missing_values = [
    "Not Available",
    "Not Applicable",
    "Not Available*",
    "N/A",
    "",
    " "
]

for df in [
    hospital_clean,
    timely_clean,
    unplanned_clean,
    infections_clean,
    complications_clean
]:
    df.replace(missing_values, np.nan, inplace=True)

In [24]:
for name, df in {
    "Hospital Information": hospital_clean,
    "Timely & Effective Care": timely_clean,
    "Unplanned Visits": unplanned_clean,
    "Infections": infections_clean,
    "Complications": complications_clean
}.items():

    print(name, ":", df.isna().sum().sum(), "missing values")

Hospital Information : 53143 missing values
Timely & Effective Care : 210064 missing values
Unplanned Visits : 310222 missing values
Infections : 274545 missing values
Complications : 259716 missing values


## Standardize Hospital Identifiers

CMS Facility IDs are standardized to six-character strings to ensure hospitals can be matched reliably across all datasets.

In [25]:
clean_datasets = {
    "Hospital Information": hospital_clean,
    "Timely & Effective Care": timely_clean,
    "Unplanned Visits": unplanned_clean,
    "Infections": infections_clean,
    "Complications": complications_clean
}

for name, df in clean_datasets.items():
    if "Facility ID" in df.columns:
        df["Facility ID"] = (
            df["Facility ID"]
            .astype("string")
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(6)
        )

## Clean Hospital Information

The hospital information dataset provides one record per hospital and will serve as the primary reference table for hospital characteristics and geographic information.

In [26]:
hospital_clean.columns.tolist()

['Facility ID',
 'Facility Name',
 'Address',
 'City/Town',
 'State',
 'ZIP Code',
 'County/Parish',
 'Telephone Number',
 'Hospital Type',
 'Hospital Ownership',
 'Emergency Services',
 'Meets criteria for birthing friendly designation',
 'Hospital overall rating',
 'Hospital overall rating footnote',
 'MORT Group Measure Count',
 'Count of Facility MORT Measures',
 'Count of MORT Measures Better',
 'Count of MORT Measures No Different',
 'Count of MORT Measures Worse',
 'MORT Group Footnote',
 'Safety Group Measure Count',
 'Count of Facility Safety Measures',
 'Count of Safety Measures Better',
 'Count of Safety Measures No Different',
 'Count of Safety Measures Worse',
 'Safety Group Footnote',
 'READM Group Measure Count',
 'Count of Facility READM Measures',
 'Count of READM Measures Better',
 'Count of READM Measures No Different',
 'Count of READM Measures Worse',
 'READM Group Footnote',
 'Pt Exp Group Measure Count',
 'Count of Facility Pt Exp Measures',
 'Pt Exp Group Footno

In [27]:
hospital_columns = [
    "Facility ID",
    "Facility Name",
    "Address",
    "City/Town",
    "State",
    "ZIP Code",
    "County/Parish",
    "Telephone Number",
    "Hospital Type",
    "Hospital Ownership",
    "Emergency Services",
    "Hospital overall rating"
]

hospital_columns = [
    col for col in hospital_columns
    if col in hospital_clean.columns
]

hospital_clean = hospital_clean[hospital_columns].copy()

## Remove Exact Duplicates

In [28]:
print("Duplicates before:", hospital_clean.duplicated().sum())

hospital_clean = hospital_clean.drop_duplicates()

print("Duplicates after:", hospital_clean.duplicated().sum())

Duplicates before: 0
Duplicates after: 0


## Create Virginia Hospital Population

The analysis focuses on UVA Medical Center relative to other Virginia hospitals. A Virginia hospital reference table is created and used to filter the quality-measure datasets.

In [29]:
virginia_hospitals = hospital_clean[hospital_clean["State"].eq("VA")].copy()

print("Virginia hospitals:", virginia_hospitals["Facility ID"].nunique())


Virginia hospitals: 96


In [30]:
va_facility_ids = set(virginia_hospitals["Facility ID"].dropna())

## Filter Quality Data to Virginia Hospitals

Only records whose CMS Facility ID belongs to a Virginia hospital are retained for the quality-measure analysis.

In [31]:
# Restrict the national quality datasets to the Virginia comparison population.
timely_clean = timely_clean[timely_clean["Facility ID"].isin(va_facility_ids)].copy()
unplanned_clean = unplanned_clean[unplanned_clean["Facility ID"].isin(va_facility_ids)].copy()
infections_clean = infections_clean[infections_clean["Facility ID"].isin(va_facility_ids)].copy()
complications_clean = complications_clean[complications_clean["Facility ID"].isin(va_facility_ids)].copy()

In [32]:
for df in [
    timely_clean,
    unplanned_clean,
    infections_clean,
    complications_clean
]:
    if "Score" in df.columns:
        df["Score Numeric"] = pd.to_numeric(
            df["Score"],
            errors="coerce"
        )

## Convert Date Fields

In [33]:
for name, df in {
    "Timely Care": timely_clean,
    "Unplanned Visitors":unplanned_clean,
    "Infections": infections_clean,
    "Complications": complications_clean }.items():
    print("\n", name)
    date_columns = [col for col in df.columns
                    if "date" in col.lower()]
    print(date_columns)


 Timely Care
['Start Date', 'End Date']

 Unplanned Visitors
['Start Date', 'End Date']

 Infections
['Start Date', 'End Date']

 Complications
['Start Date', 'End Date']


In [34]:
for df in [ timely_clean, unplanned_clean, infections_clean, complications_clean ]:
    date_columns = [col for col in df.columns if "date" in col.lower()]
    for col in date_columns: df[col] = pd.to_datetime(df[col], errors ="coerce")

## Duplicate Assessment

Exact duplicate records are removed while preserving repeated hospital records associated with different quality measures.

In [35]:
quality_datasets = {
    "Timely Care": timely_clean,
    "Unplanned Visits": unplanned_clean,
    "Infections": infections_clean,
    "Complications": complications_clean
}

for name, df in quality_datasets.items():
    before = len(df)

    df.drop_duplicates(inplace=True)

    after = len(df)

    print(
        f"{name}: {before - after} duplicate rows removed"
    )

Timely Care: 0 duplicate rows removed
Unplanned Visits: 0 duplicate rows removed
Infections: 0 duplicate rows removed
Complications: 0 duplicate rows removed




### Duplicate Check Result

No exact duplicate rows were found in the quality datasets. Repeated hospital records were retained because they represent different CMS quality measures rather than duplicate observations.


## Inspect Measure IDs

In [36]:
for name, df in quality_datasets.items():
  print("\n" + "=" * 60)
  print(name)
  print("=" * 60)
  if "Measure ID" in df.columns:
    print("Unique measures:", df["Measure ID"].nunique())


Timely Care
Unique measures: 30

Unplanned Visits
Unique measures: 14

Infections
Unique measures: 36

Complications
Unique measures: 20


In [37]:
for name, df in quality_datasets.items():

    if (
        "Measure ID" in df.columns
        and "Measure Name" in df.columns
    ):

        print("\n", name)

        display(
            df[
                ["Measure ID", "Measure Name"]
            ]
            .drop_duplicates()
            .head(30)
        )


 Timely Care


,Measure ID,Measure Name
124423,EDV,Emergency department volume
124424,GMCS,Global Malnutrition Composite Score
124425,GMCS_Malnutrition_Diagnosis_Documented,Global Malnutrition Composite Score: Malnutrit...
124426,GMCS_Malnutrition_Screening,Global Malnutrition Composite Score: Malnutrit...
124427,GMCS_Nutrition_Assessment,Global Malnutrition Composite Score: Nutrition...
124428,GMCS_Nutritional_Care_Plan,Global Malnutrition Composite Score: Nutrition...
124429,HH_HYPER,Hospital Harm - Severe Hyperglycemia
124430,HH_HYPO,Hospital Harm - Severe Hypoglycemia
124431,HH_ORAE,Hospital Harm - Opioid Related Adverse Events
124432,IMM_3,Healthcare workers given influenza vaccination



 Unplanned Visits


,Measure ID,Measure Name
60396,EDAC_30_AMI,Hospital return days for heart attack patients
60397,EDAC_30_HF,Hospital return days for heart failure patients
60398,EDAC_30_PN,Hospital return days for pneumonia patients
60399,Hybrid_HWR,Hybrid Hospital-Wide All-Cause Readmission Mea...
60400,OP_32,Rate of unplanned hospital visits after colono...
60401,OP_35_ADM,Rate of inpatient admissions for patients rece...
60402,OP_35_ED,Rate of emergency department (ED) visits for p...
60403,OP_36,Ratio of unplanned hospital visits after hospi...
60404,READM_30_AMI,Acute Myocardial Infarction (AMI) 30-Day Readm...
60405,READM_30_CABG,Rate of readmission for CABG



 Infections


,Measure ID,Measure Name
155304,HAI_1_CILOWER,Central Line Associated Bloodstream Infection ...
155305,HAI_1_CIUPPER,Central Line Associated Bloodstream Infection ...
155306,HAI_1_DOPC,Central Line Associated Bloodstream Infection:...
155307,HAI_1_ELIGCASES,Central Line Associated Bloodstream Infection ...
155308,HAI_1_NUMERATOR,Central Line Associated Bloodstream Infection ...
155309,HAI_1_SIR,Central Line Associated Bloodstream Infection ...
155310,HAI_2_CILOWER,Catheter Associated Urinary Tract Infections (...
155311,HAI_2_CIUPPER,Catheter Associated Urinary Tract Infections (...
155312,HAI_2_DOPC,Catheter Associated Urinary Tract Infections (...
155313,HAI_2_ELIGCASES,Catheter Associated Urinary Tract Infections (...



 Complications


,Measure ID,Measure Name
86280,COMP_HIP_KNEE,Rate of complications for hip/knee replacement...
86281,Hybrid_HWM,Hybrid Hospital-Wide All-Cause Risk Standardiz...
86282,MORT_30_AMI,Death rate for heart attack patients
86283,MORT_30_CABG,Death rate for CABG surgery patients
86284,MORT_30_COPD,Death rate for COPD patients
86285,MORT_30_HF,Death rate for heart failure patients
86286,MORT_30_PN,Death rate for pneumonia patients
86287,MORT_30_STK,Death rate for stroke patients
86288,PSI_03,Pressure ulcer rate
86289,PSI_04,Death rate among surgical inpatients with seri...


## Check UVA After Cleaning

In [38]:
print("Hospital Information:", (hospital_clean["Facility ID"] == uva_id).sum())

for name, df in quality_datasets.items():
    print(name,":",(df["Facility ID"] == uva_id).sum())

Hospital Information: 1
Timely Care : 30
Unplanned Visits : 14
Infections : 36
Complications : 20


## Inspect UVA's Available Measures

In [39]:
uva_timely = timely_clean[timely_clean["Facility ID"] == uva_id]

display(
    uva_timely[
        [
            col for col in [
                "Measure ID",
                "Measure Name",
                "Score",
                "Score Numeric"
            ]
            if col in uva_timely.columns
        ]
    ].head(30)
)

,Measure ID,Measure Name,Score,Score Numeric
124543,EDV,Emergency department volume,very high,NaN
124544,GMCS,Global Malnutrition Composite Score,NaN,NaN
124545,GMCS_Malnutrition_Diagnosis_Documented,Global Malnutrition Composite Score: Malnutrit...,NaN,NaN
124546,GMCS_Malnutrition_Screening,Global Malnutrition Composite Score: Malnutrit...,NaN,NaN
124547,GMCS_Nutrition_Assessment,Global Malnutrition Composite Score: Nutrition...,NaN,NaN
124548,GMCS_Nutritional_Care_Plan,Global Malnutrition Composite Score: Nutrition...,NaN,NaN
124549,HH_HYPER,Hospital Harm - Severe Hyperglycemia,9,9.0
124550,HH_HYPO,Hospital Harm - Severe Hypoglycemia,2,2.0
124551,HH_ORAE,Hospital Harm - Opioid Related Adverse Events,0,0.0
124552,IMM_3,Healthcare workers given influenza vaccination,97,97.0


In [40]:
uva_unplanned = unplanned_clean[unplanned_clean["Facility ID"] == uva_id]

display(
    uva_unplanned[
        [
            col for col in [
                "Measure ID",
                "Measure Name",
                "Score",
                "Score Numeric"
            ]
            if col in uva_unplanned.columns
        ]
    ].head(30)
)

,Measure ID,Measure Name,Score,Score Numeric
60452,EDAC_30_AMI,Hospital return days for heart attack patients,35.9,35.9
60453,EDAC_30_HF,Hospital return days for heart failure patients,-8,-8.0
60454,EDAC_30_PN,Hospital return days for pneumonia patients,-6.7,-6.7
60455,Hybrid_HWR,Hybrid Hospital-Wide All-Cause Readmission Mea...,NaN,NaN
60456,OP_32,Rate of unplanned hospital visits after colono...,10,10.0
60457,OP_35_ADM,Rate of inpatient admissions for patients rece...,11,11.0
60458,OP_35_ED,Rate of emergency department (ED) visits for p...,5.2,5.2
60459,OP_36,Ratio of unplanned hospital visits after hospi...,0.7,0.7
60460,READM_30_AMI,Acute Myocardial Infarction (AMI) 30-Day Readm...,14.3,14.3
60461,READM_30_CABG,Rate of readmission for CABG,9.6,9.6


In [41]:
uva_infections = infections_clean[infections_clean["Facility ID"] == uva_id]

display(
    uva_infections[
        [
            col for col in [
                "Measure ID",
                "Measure Name",
                "Score",
                "Score Numeric"
            ]
            if col in uva_infections.columns
        ]
    ].head(30)
)

,Measure ID,Measure Name,Score,Score Numeric
155448,HAI_1_CILOWER,Central Line Associated Bloodstream Infection ...,0.209,0.209
155449,HAI_1_CIUPPER,Central Line Associated Bloodstream Infection ...,0.625,0.625
155450,HAI_1_DOPC,Central Line Associated Bloodstream Infection:...,29956,29956.000
155451,HAI_1_ELIGCASES,Central Line Associated Bloodstream Infection ...,34.677,34.677
155452,HAI_1_NUMERATOR,Central Line Associated Bloodstream Infection ...,13,13.000
155453,HAI_1_SIR,Central Line Associated Bloodstream Infection ...,0.375,0.375
155454,HAI_2_CILOWER,Catheter Associated Urinary Tract Infections (...,0.077,0.077
155455,HAI_2_CIUPPER,Catheter Associated Urinary Tract Infections (...,0.468,0.468
155456,HAI_2_DOPC,Catheter Associated Urinary Tract Infections (...,15036,15036.000
155457,HAI_2_ELIGCASES,Catheter Associated Urinary Tract Infections (...,23.680,23.680


In [42]:
uva_complications = complications_clean[complications_clean["Facility ID"] == uva_id]

display(
    uva_complications[
        [
            col for col in [
                "Measure ID",
                "Measure Name",
                "Score",
                "Score Numeric"
            ]
            if col in uva_complications.columns
        ]
    ].head(30)
)

,Measure ID,Measure Name,Score,Score Numeric
86360,COMP_HIP_KNEE,Rate of complications for hip/knee replacement...,4.3,4.30
86361,Hybrid_HWM,Hybrid Hospital-Wide All-Cause Risk Standardiz...,3.3,3.30
86362,MORT_30_AMI,Death rate for heart attack patients,10.4,10.40
86363,MORT_30_CABG,Death rate for CABG surgery patients,2,2.00
86364,MORT_30_COPD,Death rate for COPD patients,9.5,9.50
86365,MORT_30_HF,Death rate for heart failure patients,10.2,10.20
86366,MORT_30_PN,Death rate for pneumonia patients,14.7,14.70
86367,MORT_30_STK,Death rate for stroke patients,13.2,13.20
86368,PSI_03,Pressure ulcer rate,0.45,0.45
86369,PSI_04,Death rate among surgical inpatients with seri...,141.48,141.48


## Check Numeric Score Coverage

In [43]:
for name, df in quality_datasets.items():

    if "Score Numeric" in df.columns:

        numeric_count = df["Score Numeric"].notna().sum()
        total = len(df)

        print(
            f"{name}: {numeric_count:,} numeric scores "
            f"out of {total:,} rows"
        )

Timely Care: 1,213 numeric scores out of 2,492 rows
Unplanned Visits: 730 numeric scores out of 1,218 rows
Infections: 2,070 numeric scores out of 3,132 rows
Complications: 1,234 numeric scores out of 1,740 rows


## Update Clean Dataset Dictionary

In [44]:
clean_datasets = {
    "Hospital Information": hospital_clean,
    "Timely & Effective Care": timely_clean,
    "Unplanned Visits": unplanned_clean,
    "Infections": infections_clean,
    "Complications & Mortality": complications_clean
}

## Cleaning Summary

The five CMS hospital datasets were standardized for analysis. Missing-value labels were converted to standard null values, hospital identifiers were standardized, and date fields were converted to datetime format.

The hospital information dataset was reduced to relevant identifying and organizational characteristics. Virginia hospitals were isolated to create the comparison population for UVA Medical Center.

The four quality-measure datasets were then filtered using Virginia CMS Facility IDs. Numeric versions of CMS quality scores were created while preserving the original reported values.

Exact duplicate records were removed without removing legitimate repeated hospital records associated with different quality measures.

The resulting datasets remain in long format, where each hospital may have multiple records corresponding to different quality measures. This structure will support hospital-level benchmarking and relational analysis in later stages of the project.

---

# Day 3: Analysis Preparation and Benchmarking

This stage narrows the cleaned CMS data to a focused set of hospital-quality measures and prepares an analysis-ready table for UVA Medical Center and Virginia peer hospitals.

The analysis uses 12 selected measures across four areas: timely care, readmissions/unplanned visits, healthcare-associated infections, and mortality/complications.

In [45]:
for name, df in {
    "Timely Care": timely_clean,
    "Unplanned Visits": unplanned_clean,
    "Infections": infections_clean,
    "Complications": complications_clean
}.items():

    print("\n", name)

    display(
        df[
            ["Measure ID", "Measure Name"]
        ]
        .drop_duplicates()
        .head(30)
    )


 Timely Care


,Measure ID,Measure Name
124423,EDV,Emergency department volume
124424,GMCS,Global Malnutrition Composite Score
124425,GMCS_Malnutrition_Diagnosis_Documented,Global Malnutrition Composite Score: Malnutrit...
124426,GMCS_Malnutrition_Screening,Global Malnutrition Composite Score: Malnutrit...
124427,GMCS_Nutrition_Assessment,Global Malnutrition Composite Score: Nutrition...
124428,GMCS_Nutritional_Care_Plan,Global Malnutrition Composite Score: Nutrition...
124429,HH_HYPER,Hospital Harm - Severe Hyperglycemia
124430,HH_HYPO,Hospital Harm - Severe Hypoglycemia
124431,HH_ORAE,Hospital Harm - Opioid Related Adverse Events
124432,IMM_3,Healthcare workers given influenza vaccination



 Unplanned Visits


,Measure ID,Measure Name
60396,EDAC_30_AMI,Hospital return days for heart attack patients
60397,EDAC_30_HF,Hospital return days for heart failure patients
60398,EDAC_30_PN,Hospital return days for pneumonia patients
60399,Hybrid_HWR,Hybrid Hospital-Wide All-Cause Readmission Mea...
60400,OP_32,Rate of unplanned hospital visits after colono...
60401,OP_35_ADM,Rate of inpatient admissions for patients rece...
60402,OP_35_ED,Rate of emergency department (ED) visits for p...
60403,OP_36,Ratio of unplanned hospital visits after hospi...
60404,READM_30_AMI,Acute Myocardial Infarction (AMI) 30-Day Readm...
60405,READM_30_CABG,Rate of readmission for CABG



 Infections


,Measure ID,Measure Name
155304,HAI_1_CILOWER,Central Line Associated Bloodstream Infection ...
155305,HAI_1_CIUPPER,Central Line Associated Bloodstream Infection ...
155306,HAI_1_DOPC,Central Line Associated Bloodstream Infection:...
155307,HAI_1_ELIGCASES,Central Line Associated Bloodstream Infection ...
155308,HAI_1_NUMERATOR,Central Line Associated Bloodstream Infection ...
155309,HAI_1_SIR,Central Line Associated Bloodstream Infection ...
155310,HAI_2_CILOWER,Catheter Associated Urinary Tract Infections (...
155311,HAI_2_CIUPPER,Catheter Associated Urinary Tract Infections (...
155312,HAI_2_DOPC,Catheter Associated Urinary Tract Infections (...
155313,HAI_2_ELIGCASES,Catheter Associated Urinary Tract Infections (...



 Complications


,Measure ID,Measure Name
86280,COMP_HIP_KNEE,Rate of complications for hip/knee replacement...
86281,Hybrid_HWM,Hybrid Hospital-Wide All-Cause Risk Standardiz...
86282,MORT_30_AMI,Death rate for heart attack patients
86283,MORT_30_CABG,Death rate for CABG surgery patients
86284,MORT_30_COPD,Death rate for COPD patients
86285,MORT_30_HF,Death rate for heart failure patients
86286,MORT_30_PN,Death rate for pneumonia patients
86287,MORT_30_STK,Death rate for stroke patients
86288,PSI_03,Pressure ulcer rate
86289,PSI_04,Death rate among surgical inpatients with seri...


## Select Three Measures per Category

Three measures are selected from each quality category to create a focused 12-measure analysis.

In [46]:
selected_timely_ids = [
    "OP_18d",
    "SAFE_USE_OF_OPIOIDS",
    "OP_22"
]

selected_unplanned_ids = [
    "OP_36",
    "READM_30_HF",
    "OP_35_ADM"
]

selected_infection_ids = [
    "HAI_1_SIR",
    "HAI_2_SIR",
    "HAI_3_SIR"
]

selected_complication_ids = [
    "MORT_30_STK",
    "PSI_04",
    "PSI_08"
]


In [47]:
# Create focused analysis tables from the selected CMS measure IDs.
timely_analysis = timely_clean[
    timely_clean["Measure ID"].isin(selected_timely_ids)
].copy()

unplanned_analysis = unplanned_clean[
    unplanned_clean["Measure ID"].isin(selected_unplanned_ids)
].copy()

infections_analysis = infections_clean[
    infections_clean["Measure ID"].isin(selected_infection_ids)
].copy()

complications_analysis = complications_clean[
    complications_clean["Measure ID"].isin(selected_complication_ids)
].copy()

In [48]:
# Label each selected measure with its broader quality category.
timely_analysis["Quality Category"] = "Timely Care"
unplanned_analysis["Quality Category"] = "Readmissions"
infections_analysis["Quality Category"] = "Infections"
complications_analysis["Quality Category"] = "Mortality & Complications"

In [49]:
analysis_sets = {
    "Timely Care": timely_analysis,
    "Readmissions": unplanned_analysis,
    "Infections": infections_analysis,
    "Mortality & Complications": complications_analysis
}

for name, df in analysis_sets.items():

    print("\n", name)

    # Make sure Facility ID is text
    df["Facility ID"] = df["Facility ID"].astype(str).str.strip()

    # Filter UVA rows
    uva_rows = df[df["Facility ID"] == uva_id]

    columns_to_show = [
        col for col in [
            "Measure ID",
            "Measure Name",
            "Score",
            "Score Numeric"
        ]
        if col in uva_rows.columns
    ]

    display(uva_rows[columns_to_show])




 Timely Care


,Measure ID,Measure Name,Score,Score Numeric
124556,OP_18d,Average (median) time patients spent in the em...,446,446.0
124557,OP_22,Left before being seen,2,2.0
124562,SAFE_USE_OF_OPIOIDS,Safe Use of Opioids - Concurrent Prescribing,10,10.0



 Readmissions


,Measure ID,Measure Name,Score,Score Numeric
60457,OP_35_ADM,Rate of inpatient admissions for patients rece...,11,11.0
60459,OP_36,Ratio of unplanned hospital visits after hospi...,0.7,0.7
60463,READM_30_HF,Heart failure (HF) 30-Day Readmission Rate,19.9,19.9



 Infections


,Measure ID,Measure Name,Score,Score Numeric
155453,HAI_1_SIR,Central Line Associated Bloodstream Infection ...,0.375,0.375
155459,HAI_2_SIR,Catheter Associated Urinary Tract Infections (...,0.211,0.211
155465,HAI_3_SIR,SSI - Colon Surgery,0.900,0.900



 Mortality & Complications


,Measure ID,Measure Name,Score,Score Numeric
86367,MORT_30_STK,Death rate for stroke patients,13.2,13.20
86369,PSI_04,Death rate among surgical inpatients with seri...,141.48,141.48
86371,PSI_08,In-hospital fall-associated fracture rate,0.37,0.37


## Combine the Four Quality Tables

The selected quality datasets are standardized to a common column structure and stacked into one analysis-ready table for benchmarking.

In [50]:
columns_to_keep = [
    "Facility ID",
    "Facility Name",
    "Measure ID",
    "Measure Name",
    "Score Numeric",
    "Quality Category"
]

analysis_tables = [
    timely_analysis,
    unplanned_analysis,
    infections_analysis,
    complications_analysis
]

prepared_tables = []

for df in analysis_tables:

    available_columns = [
        col for col in columns_to_keep
        if col in df.columns
    ]

    prepared_tables.append(
        df[available_columns].copy()
    )

hospital_quality_analysis = pd.concat(
    prepared_tables,
    ignore_index=True
)

In [51]:
# Flag UVA records so the focal hospital can be separated from the peer population.
hospital_quality_analysis["Is UVA"] = (
    hospital_quality_analysis["Facility ID"] == uva_id
)

hospital_quality_analysis.head()

,Facility ID,Facility Name,Measure ID,Measure Name,Score Numeric,Quality Category,Is UVA
0,490002,RUSSELL COUNTY HOSPITAL,OP_18d,Average (median) time patients spent in the em...,NaN,Timely Care,False
1,490002,RUSSELL COUNTY HOSPITAL,OP_22,Left before being seen,1.0,Timely Care,False
2,490002,RUSSELL COUNTY HOSPITAL,SAFE_USE_OF_OPIOIDS,Safe Use of Opioids - Concurrent Prescribing,16.0,Timely Care,False
3,490004,SENTARA RMH MEDICAL CENTER,OP_18d,Average (median) time patients spent in the em...,301.0,Timely Care,False
4,490004,SENTARA RMH MEDICAL CENTER,OP_22,Left before being seen,3.0,Timely Care,False


## Inspect the Finished Analysis Table

In [52]:
hospital_quality_analysis["Quality Category"].value_counts()

,count
Quality Category,
Readmissions,261
Infections,261
Mortality & Complications,261
Timely Care,248


## Calculate Virginia Peer Benchmarks

UVA is excluded from the benchmark population so its own score does not influence the baseline used for comparison.

In [53]:
# Exclude UVA from the peer benchmark population.
peer_quality = hospital_quality_analysis[
    ~hospital_quality_analysis["Is UVA"]
].copy()

va_benchmarks = (
    peer_quality
    .groupby(
        ["Measure ID", "Measure Name"],
        as_index=False
    )["Score Numeric"]
    .agg(
        Virginia_Peer_Mean="mean",
        Virginia_Peer_Median="median",
        Peer_Hospital_Count="count"
    )
)

va_benchmarks

,Measure ID,Measure Name,Virginia_Peer_Mean,Virginia_Peer_Median,Peer_Hospital_Count
0,HAI_1_SIR,Central Line Associated Bloodstream Infection ...,0.372023,0.3375,44
1,HAI_2_SIR,Catheter Associated Urinary Tract Infections (...,0.373796,0.3070,49
2,HAI_3_SIR,SSI - Colon Surgery,0.733477,0.6705,44
3,MORT_30_STK,Death rate for stroke patients,11.630769,11.8000,65
4,OP_18d,Average (median) time patients spent in the em...,356.382979,346.0000,47
5,OP_22,Left before being seen,1.670886,1.0000,79
6,OP_35_ADM,Rate of inpatient admissions for patients rece...,10.864516,10.8000,31
7,OP_36,Ratio of unplanned hospital visits after hospi...,1.028125,1.0000,64
8,PSI_04,Death rate among surgical inpatients with seri...,168.204000,167.6350,40
9,PSI_08,In-hospital fall-associated fracture rate,0.262361,0.2600,72


## Create the UVA Comparison Table

UVA's reported score for each selected measure is joined to the peer benchmark so the focal hospital can be compared with the mean and median of other reporting Virginia hospitals.

In [54]:
uva_scores = hospital_quality_analysis[
    hospital_quality_analysis["Is UVA"]
][
    [
        "Measure ID",
        "Measure Name",
        "Quality Category",
        "Score Numeric"
    ]
].copy()

uva_comparison = uva_scores.merge(
    va_benchmarks,
    on=["Measure ID", "Measure Name"],
    how="left"
)

uva_comparison["Difference_From_VA_Peer_Mean"] = (
    uva_comparison["Score Numeric"]
    - uva_comparison["Virginia_Peer_Mean"]
)

uva_comparison

,Measure ID,Measure Name,Quality Category,Score Numeric,Virginia_Peer_Mean,Virginia_Peer_Median,Peer_Hospital_Count,Difference_From_VA_Peer_Mean
0,OP_18d,Average (median) time patients spent in the em...,Timely Care,446.000,356.382979,346.0000,47,89.617021
1,OP_22,Left before being seen,Timely Care,2.000,1.670886,1.0000,79,0.329114
2,SAFE_USE_OF_OPIOIDS,Safe Use of Opioids - Concurrent Prescribing,Timely Care,10.000,14.120000,13.0000,75,-4.120000
3,OP_35_ADM,Rate of inpatient admissions for patients rece...,Readmissions,11.000,10.864516,10.8000,31,0.135484
4,OP_36,Ratio of unplanned hospital visits after hospi...,Readmissions,0.700,1.028125,1.0000,64,-0.328125
5,READM_30_HF,Heart failure (HF) 30-Day Readmission Rate,Readmissions,19.900,21.246753,21.3000,77,-1.346753
6,HAI_1_SIR,Central Line Associated Bloodstream Infection ...,Infections,0.375,0.372023,0.3375,44,0.002977
7,HAI_2_SIR,Catheter Associated Urinary Tract Infections (...,Infections,0.211,0.373796,0.3070,49,-0.162796
8,HAI_3_SIR,SSI - Colon Surgery,Infections,0.900,0.733477,0.6705,44,0.166523
9,MORT_30_STK,Death rate for stroke patients,Mortality & Complications,13.200,11.630769,11.8000,65,1.569231


In [55]:
hospital_quality_analysis.to_csv(
    "hospital_quality_analysis.csv",
    index=False
)

uva_comparison.to_csv(
    "uva_vs_virginia_peers.csv",
    index=False
)

## Day 3 Summary

A focused set of 12 hospital-quality measures was selected across timely care, readmissions, infections, and mortality/complications. The selected Virginia hospital records were combined into a single analysis-ready dataset, with UVA Medical Center flagged as the focal hospital.

Virginia peer means and medians were then calculated after excluding UVA from the benchmark population. The resulting comparison table provides measure-level UVA scores, peer benchmarks, and raw differences for later SQL validation and Power BI visualization.

---

# Day 4: SQL Analysis and Advanced Benchmarking

The analysis-ready hospital data is loaded into a SQLite database for relational analysis and benchmark validation. SQL is used to retrieve UVA records, calculate peer benchmarks, join hospital characteristics, classify hospitals, test join integrity, and compare UVA with other reporting Virginia hospitals.

In [56]:
import sqlite3
conn = sqlite3.connect("uva_hospital_analysis.db")

In [57]:
hospital_clean.to_sql(
    "hospitals",
    conn,
    if_exists="replace",
    index=False
)

hospital_quality_analysis.to_sql(
    "hospital_quality",
    conn,
    if_exists="replace",
    index=False
)

uva_comparison.to_sql(
    "uva_benchmark",
    conn,
    if_exists="replace",
    index=False
)

12

In [58]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

pd.read_sql_query(query, conn)

,name
0,hospitals
1,hospital_quality
2,uva_benchmark


In [59]:
pd.read_sql_query("SELECT * FROM hospital_quality Limit 5;", conn)

,Facility ID,Facility Name,Measure ID,Measure Name,Score Numeric,Quality Category,Is UVA
0,490002,RUSSELL COUNTY HOSPITAL,OP_18d,Average (median) time patients spent in the em...,NaN,Timely Care,0
1,490002,RUSSELL COUNTY HOSPITAL,OP_22,Left before being seen,1.0,Timely Care,0
2,490002,RUSSELL COUNTY HOSPITAL,SAFE_USE_OF_OPIOIDS,Safe Use of Opioids - Concurrent Prescribing,16.0,Timely Care,0
3,490004,SENTARA RMH MEDICAL CENTER,OP_18d,Average (median) time patients spent in the em...,301.0,Timely Care,0
4,490004,SENTARA RMH MEDICAL CENTER,OP_22,Left before being seen,3.0,Timely Care,0


## UVA Medical Center Quality Measures

This query retrieves the selected quality measures reported for UVA Medical Center

In [60]:
pd.read_sql_query("""
SELECT
    "Quality Category",
    "Measure Name",
    "Score Numeric"
FROM hospital_quality
WHERE "Facility ID" = '490009'
ORDER BY "Quality Category";
""", conn)



,Quality Category,Measure Name,Score Numeric
0,Infections,Central Line Associated Bloodstream Infection ...,0.375
1,Infections,Catheter Associated Urinary Tract Infections (...,0.211
2,Infections,SSI - Colon Surgery,0.900
3,Mortality & Complications,Death rate for stroke patients,13.200
4,Mortality & Complications,Death rate among surgical inpatients with seri...,141.480
5,Mortality & Complications,In-hospital fall-associated fracture rate,0.370
6,Readmissions,Rate of inpatient admissions for patients rece...,11.000
7,Readmissions,Ratio of unplanned hospital visits after hospi...,0.700
8,Readmissions,Heart failure (HF) 30-Day Readmission Rate,19.900
9,Timely Care,Average (median) time patients spent in the em...,446.000


## Virginia Peer Quality Benchmarks

This query calculates the average reported score and number of peer hospitals contributing data for each selected quality measure. UVA is excluded from the benchmark population.

In [61]:
query = """
SELECT
    "Measure ID",
    "Measure Name",
    AVG("Score Numeric") AS Virginia_Peer_Average,
    COUNT("Score Numeric") AS Peer_Hospital_Count

FROM hospital_quality

WHERE "Score Numeric" IS NOT NULL
  AND "Facility ID" <> '490009'

GROUP BY
    "Measure ID",
    "Measure Name"

ORDER BY
    "Measure Name";
"""

va_sql_benchmarks = pd.read_sql_query(query, conn)

va_sql_benchmarks

,Measure ID,Measure Name,Virginia_Peer_Average,Peer_Hospital_Count
0,OP_18d,Average (median) time patients spent in the em...,356.382979,47
1,HAI_2_SIR,Catheter Associated Urinary Tract Infections (...,0.373796,49
2,HAI_1_SIR,Central Line Associated Bloodstream Infection ...,0.372023,44
3,PSI_04,Death rate among surgical inpatients with seri...,168.204000,40
4,MORT_30_STK,Death rate for stroke patients,11.630769,65
5,READM_30_HF,Heart failure (HF) 30-Day Readmission Rate,21.246753,77
6,PSI_08,In-hospital fall-associated fracture rate,0.262361,72
7,OP_22,Left before being seen,1.670886,79
8,OP_35_ADM,Rate of inpatient admissions for patients rece...,10.864516,31
9,OP_36,Ratio of unplanned hospital visits after hospi...,1.028125,64


## Quality Category Coverage

This query summarizes the number of records, hospitals, and measures represented within each quality category.

In [62]:
query = """
SELECT
    "Quality Category",
    COUNT(*) AS Records,
    COUNT(DISTINCT "Facility ID") AS Hospitals,
    COUNT(DISTINCT "Measure ID") AS Measures

FROM hospital_quality

GROUP BY "Quality Category"

ORDER BY Records DESC;
"""

category_summary = pd.read_sql_query(query, conn)

display(category_summary)

,Quality Category,Records,Hospitals,Measures
0,Readmissions,261,87,3
1,Mortality & Complications,261,87,3
2,Infections,261,87,3
3,Timely Care,248,84,3


## Join Hospital Characteristics with Quality Measures

Hospital quality records are connected with hospital characteristics using the CMS Facility ID as the common relational key.

In [63]:
query = """
SELECT
    q."Facility ID",
    h."Facility Name",
    h."City/Town",
    h."Hospital Type",
    h."Hospital Ownership",
    q."Quality Category",
    q."Measure Name",
    q."Score Numeric"

FROM hospital_quality AS q

LEFT JOIN hospitals AS h
    ON q."Facility ID" = h."Facility ID"

WHERE q."Score Numeric" IS NOT NULL;
"""

joined_quality = pd.read_sql_query(query, conn)

display(joined_quality.head())


,Facility ID,Facility Name,City/Town,Hospital Type,Hospital Ownership,Quality Category,Measure Name,Score Numeric
0,490002,RUSSELL COUNTY HOSPITAL,LEBANON,Acute Care Hospitals,Voluntary non-profit - Private,Timely Care,Left before being seen,1.0
1,490002,RUSSELL COUNTY HOSPITAL,LEBANON,Acute Care Hospitals,Voluntary non-profit - Private,Timely Care,Safe Use of Opioids - Concurrent Prescribing,16.0
2,490004,SENTARA RMH MEDICAL CENTER,HARRISONBURG,Acute Care Hospitals,Voluntary non-profit - Private,Timely Care,Average (median) time patients spent in the em...,301.0
3,490004,SENTARA RMH MEDICAL CENTER,HARRISONBURG,Acute Care Hospitals,Voluntary non-profit - Private,Timely Care,Left before being seen,3.0
4,490004,SENTARA RMH MEDICAL CENTER,HARRISONBURG,Acute Care Hospitals,Voluntary non-profit - Private,Timely Care,Safe Use of Opioids - Concurrent Prescribing,15.0


## Hospital Group Classification

A CASE statement identifies UVA Medical Center separately from other Virginia hospitals for comparison.

In [64]:
query = """
SELECT
    "Facility ID",
    "Facility Name",
    "Measure Name",
    "Score Numeric",

    CASE
        WHEN "Facility ID" = '490009'
        THEN 'UVA Medical Center'
        ELSE 'Other Virginia Hospital'
    END AS Hospital_Group

FROM hospital_quality

WHERE "Score Numeric" IS NOT NULL;
"""

hospital_groups = pd.read_sql_query(query, conn)

display(hospital_groups.head(10))

,Facility ID,Facility Name,Measure Name,Score Numeric,Hospital_Group
0,490002,RUSSELL COUNTY HOSPITAL,Left before being seen,1.0,Other Virginia Hospital
1,490002,RUSSELL COUNTY HOSPITAL,Safe Use of Opioids - Concurrent Prescribing,16.0,Other Virginia Hospital
2,490004,SENTARA RMH MEDICAL CENTER,Average (median) time patients spent in the em...,301.0,Other Virginia Hospital
3,490004,SENTARA RMH MEDICAL CENTER,Left before being seen,3.0,Other Virginia Hospital
4,490004,SENTARA RMH MEDICAL CENTER,Safe Use of Opioids - Concurrent Prescribing,15.0,Other Virginia Hospital
5,490005,WINCHESTER MEDICAL CENTER,Left before being seen,3.0,Other Virginia Hospital
6,490005,WINCHESTER MEDICAL CENTER,Safe Use of Opioids - Concurrent Prescribing,13.0,Other Virginia Hospital
7,490007,SENTARA NORFOLK GENERAL HOSPITAL,Left before being seen,4.0,Other Virginia Hospital
8,490007,SENTARA NORFOLK GENERAL HOSPITAL,Safe Use of Opioids - Concurrent Prescribing,11.0,Other Virginia Hospital
9,490009,UNIVERSITY OF VIRGINIA MEDICAL CENTER,Average (median) time patients spent in the em...,446.0,UVA Medical Center


## UVA vs Virginia Peer Benchmark Using a CTE

A common table expression calculates the peer average for each measure after excluding UVA, then joins those benchmarks back to UVA's reported scores.

In [65]:
query = """
WITH VirginiaPeerBenchmark AS (

    SELECT
        "Measure ID",
        "Measure Name",
        AVG("Score Numeric") AS Virginia_Peer_Average

    FROM hospital_quality

    WHERE "Score Numeric" IS NOT NULL
      AND "Facility ID" <> '490009'

    GROUP BY
        "Measure ID",
        "Measure Name"
)

SELECT
    q."Quality Category",
    q."Measure Name",
    q."Score Numeric" AS UVA_Score,
    v.Virginia_Peer_Average,

    q."Score Numeric" - v.Virginia_Peer_Average
        AS Difference_From_VA_Peer_Average

FROM hospital_quality AS q

INNER JOIN VirginiaPeerBenchmark AS v
    ON q."Measure ID" = v."Measure ID"

WHERE q."Facility ID" = '490009'

ORDER BY q."Quality Category";
"""

uva_vs_va_sql = pd.read_sql_query(query, conn)

uva_vs_va_sql

,Quality Category,Measure Name,UVA_Score,Virginia_Peer_Average,Difference_From_VA_Peer_Average
0,Infections,Central Line Associated Bloodstream Infection ...,0.375,0.372023,0.002977
1,Infections,Catheter Associated Urinary Tract Infections (...,0.211,0.373796,-0.162796
2,Infections,SSI - Colon Surgery,0.900,0.733477,0.166523
3,Mortality & Complications,Death rate for stroke patients,13.200,11.630769,1.569231
4,Mortality & Complications,Death rate among surgical inpatients with seri...,141.480,168.204000,-26.724000
5,Mortality & Complications,In-hospital fall-associated fracture rate,0.370,0.262361,0.107639
6,Readmissions,Rate of inpatient admissions for patients rece...,11.000,10.864516,0.135484
7,Readmissions,Ratio of unplanned hospital visits after hospi...,0.700,1.028125,-0.328125
8,Readmissions,Heart failure (HF) 30-Day Readmission Rate,19.900,21.246753,-1.346753
9,Timely Care,Average (median) time patients spent in the em...,446.000,356.382979,89.617021


## Measure-Level Peer Window Analysis

A window function calculates the peer average for each measure while preserving individual hospital records. UVA remains visible as a row, but its score is excluded from the peer-average calculation.

In [66]:
query = """
SELECT
    "Facility ID",
    "Facility Name",
    "Measure Name",
    "Score Numeric",

    AVG(
        CASE
            WHEN "Facility ID" <> '490009'
            THEN "Score Numeric"
        END
    ) OVER (
        PARTITION BY "Measure ID"
    ) AS Virginia_Peer_Average

FROM hospital_quality

WHERE "Score Numeric" IS NOT NULL

ORDER BY
    "Measure Name",
    "Score Numeric";
"""

window_analysis = pd.read_sql_query(query, conn)

window_analysis.head(20)

,Facility ID,Facility Name,Measure Name,Score Numeric,Virginia_Peer_Average
0,490018,AUGUSTA HEALTH,Average (median) time patients spent in the em...,235.0,356.382979
1,490077,SENTARA MARTHA JEFFERSON HOSPITAL,Average (median) time patients spent in the em...,249.0,356.382979
2,490046,SENTARA LEIGH HOSPITAL,Average (median) time patients spent in the em...,262.0,356.382979
3,490066,SENTARA WILLIAMSBURG REGIONAL MEDICAL CENTER,Average (median) time patients spent in the em...,263.0,356.382979
4,490037,RIVERSIDE SHORE MEMORIAL HOSPITAL,Average (median) time patients spent in the em...,272.0,356.382979
5,490145,STONESPRINGS HOSPITAL CENTER,Average (median) time patients spent in the em...,285.0,356.382979
6,490111,WYTHE COUNTY COMMUNITY HOSPITAL,Average (median) time patients spent in the em...,288.0,356.382979
7,490122,INOVA MOUNT VERNON HOSPITAL,Average (median) time patients spent in the em...,291.0,356.382979
8,491303,DICKENSON COMMUNITY HOSPITAL,Average (median) time patients spent in the em...,294.0,356.382979
9,491308,RAPPAHANNOCK GENERAL HOSPITAL,Average (median) time patients spent in the em...,295.0,356.382979


## Core SQL Analysis Summary

The cleaned hospital-quality data was loaded into SQLite and queried using filtering, aggregation, joins, conditional logic, CTEs, and a window function. UVA was separated from the peer benchmark population so peer averages represent other reporting Virginia hospitals rather than a statewide average that includes the focal hospital.

The following section extends the SQL work with validation, standardized scores, performance-direction logic, and benchmark-status classification.

---

# Advanced Benchmark Validation

## Join Integrity Test

This test validates the relationship between the hospital-quality and hospital-information tables. It checks for unmatched quality records, duplicated hospital identifiers, and unexpected row increases after the join.

In [67]:
query = """
SELECT

    (SELECT COUNT(*)
     FROM hospital_quality) AS Quality_Rows,

    (SELECT COUNT(*)
     FROM hospital_quality q
     LEFT JOIN hospitals h
         ON q."Facility ID" = h."Facility ID"
    ) AS Joined_Rows,

    (SELECT COUNT(*)
     FROM hospital_quality q
     LEFT JOIN hospitals h
         ON q."Facility ID" = h."Facility ID"
     WHERE h."Facility ID" IS NULL
    ) AS Unmatched_Rows,

    (SELECT COUNT(*)
     FROM (
         SELECT "Facility ID"
         FROM hospitals
         GROUP BY "Facility ID"
         HAVING COUNT(*) > 1
     )
    ) AS Duplicate_Hospital_IDs;
"""

join_integrity = pd.read_sql_query(query, conn)

join_integrity

,Quality_Rows,Joined_Rows,Unmatched_Rows,Duplicate_Hospital_IDs
0,1031,1031,0,0


## Interpreting Z-Scores

A raw z-score describes how far a score is from the Virginia peer mean in standard-deviation units:

- **Z ≈ 0:** close to the peer mean
- **Positive raw Z:** numerically above the peer mean
- **Negative raw Z:** numerically below the peer mean
- **Larger |Z|:** more unusual relative to peers

A raw z-score does **not** automatically mean better or worse because some hospital measures are better when lower. Performance direction is incorporated in a later step.

In [68]:
conn.create_function("SQRT", 1, lambda x: None if x is None or x < 0 else x ** 0.5)

## UVA Peer Z-Score Analysis

Z-scores standardize UVA's scores relative to other reporting Virginia hospitals. UVA is excluded from the mean and standard-deviation calculations so the focal hospital does not influence its own benchmark.

In [69]:
query = """
WITH MeasureStats AS (

    SELECT
        "Measure ID",
        AVG("Score Numeric") AS VA_Peer_Mean,
        AVG("Score Numeric" * "Score Numeric") AS Mean_Squared,
        COUNT("Score Numeric") AS Peer_Hospital_Count

    FROM hospital_quality

    WHERE "Score Numeric" IS NOT NULL
      AND "Facility ID" <> '490009'

    GROUP BY "Measure ID"
),

StatsWithSD AS (

    SELECT
        "Measure ID",
        VA_Peer_Mean,
        Peer_Hospital_Count,

        SQRT(
            MAX(
                Mean_Squared - (VA_Peer_Mean * VA_Peer_Mean),
                0
            )
        ) AS VA_Peer_StdDev

    FROM MeasureStats
)

SELECT
    q."Quality Category",
    q."Measure ID",
    q."Measure Name",
    q."Score Numeric" AS UVA_Score,
    s.VA_Peer_Mean,
    s.VA_Peer_StdDev,

    ROUND(
        (
            q."Score Numeric" - s.VA_Peer_Mean
        ) / NULLIF(s.VA_Peer_StdDev, 0),
        3
    ) AS Raw_Z_Score

FROM hospital_quality q

JOIN StatsWithSD s
    ON q."Measure ID" = s."Measure ID"

WHERE q."Facility ID" = '490009'
  AND q."Score Numeric" IS NOT NULL

ORDER BY ABS(Raw_Z_Score) DESC;
"""

uva_zscores = pd.read_sql_query(query, conn)

uva_zscores

,Quality Category,Measure ID,Measure Name,UVA_Score,VA_Peer_Mean,VA_Peer_StdDev,Raw_Z_Score
0,Mortality & Complications,PSI_08,In-hospital fall-associated fracture rate,0.370,0.262361,0.033061,3.256
1,Readmissions,OP_36,Ratio of unplanned hospital visits after hospi...,0.700,1.028125,0.102269,-3.208
2,Mortality & Complications,PSI_04,Death rate among surgical inpatients with seri...,141.480,168.204000,19.112151,-1.398
3,Timely Care,OP_18d,Average (median) time patients spent in the em...,446.000,356.382979,71.225609,1.258
4,Timely Care,SAFE_USE_OF_OPIOIDS,Safe Use of Opioids - Concurrent Prescribing,10.000,14.120000,3.448517,-1.195
5,Readmissions,READM_30_HF,Heart failure (HF) 30-Day Readmission Rate,19.900,21.246753,1.311348,-1.027
6,Mortality & Complications,MORT_30_STK,Death rate for stroke patients,13.200,11.630769,1.898900,0.826
7,Infections,HAI_2_SIR,Catheter Associated Urinary Tract Infections (...,0.211,0.373796,0.368292,-0.442
8,Timely Care,OP_22,Left before being seen,2.000,1.670886,1.198327,0.275
9,Infections,HAI_3_SIR,SSI - Colon Surgery,0.900,0.733477,0.604544,0.275


## Performance Direction Logic

Each selected measure is assigned a performance direction so numerical differences can be interpreted consistently. For the 12 measures selected in this project, lower reported values represent more favorable performance.

In [70]:
direction_data = {
    "Measure ID": [
        "OP_18d",
        "SAFE_USE_OF_OPIOIDS",
        "OP_22",
        "OP_36",
        "READM_30_HF",
        "OP_35_ADM",
        "HAI_1_SIR",
        "HAI_2_SIR",
        "HAI_3_SIR",
        "MORT_30_STK",
        "PSI_04",
        "PSI_08"
    ],
    "Performance Direction": [
        "LOWER",
        "LOWER",
        "LOWER",
        "LOWER",
        "LOWER",
        "LOWER",
        "LOWER",
        "LOWER",
        "LOWER",
        "LOWER",
        "LOWER",
        "LOWER"
    ]
}

measure_direction = pd.DataFrame(direction_data)

measure_direction

,Measure ID,Performance Direction
0,OP_18d,LOWER
1,SAFE_USE_OF_OPIOIDS,LOWER
2,OP_22,LOWER
3,OP_36,LOWER
4,READM_30_HF,LOWER
5,OP_35_ADM,LOWER
6,HAI_1_SIR,LOWER
7,HAI_2_SIR,LOWER
8,HAI_3_SIR,LOWER
9,MORT_30_STK,LOWER


## Direction-Adjusted Peer Benchmark Status

Measure-specific performance direction is applied before classifying UVA relative to the Virginia peer average. This prevents a lower-is-better measure from being interpreted backward.

In [71]:
measure_direction.to_sql(
    "measure_direction",
    conn,
    if_exists="replace",
    index=False
)

12

In [72]:
query = """
WITH VirginiaPeerBenchmark AS (

    SELECT
        "Measure ID",
        AVG("Score Numeric") AS Virginia_Peer_Average

    FROM hospital_quality

    WHERE "Score Numeric" IS NOT NULL
      AND "Facility ID" <> '490009'

    GROUP BY "Measure ID"
)

SELECT
    q."Quality Category",
    q."Measure ID",
    q."Measure Name",
    q."Score Numeric" AS UVA_Score,
    v.Virginia_Peer_Average,
    d."Performance Direction",

    ROUND(
        q."Score Numeric" - v.Virginia_Peer_Average,
        3
    ) AS Raw_Difference,

    ROUND(
        CASE
            WHEN d."Performance Direction" = 'LOWER'
                THEN v.Virginia_Peer_Average - q."Score Numeric"
            WHEN d."Performance Direction" = 'HIGHER'
                THEN q."Score Numeric" - v.Virginia_Peer_Average
        END,
        3
    ) AS Direction_Adjusted_Gap,

    CASE
        WHEN d."Performance Direction" = 'LOWER'
             AND q."Score Numeric" < v.Virginia_Peer_Average
            THEN 'Better Than VA Peer Average'
        WHEN d."Performance Direction" = 'HIGHER'
             AND q."Score Numeric" > v.Virginia_Peer_Average
            THEN 'Better Than VA Peer Average'
        WHEN q."Score Numeric" = v.Virginia_Peer_Average
            THEN 'Equal to VA Peer Average'
        ELSE 'Worse Than VA Peer Average'
    END AS Benchmark_Status

FROM hospital_quality q

JOIN VirginiaPeerBenchmark v
    ON q."Measure ID" = v."Measure ID"

LEFT JOIN measure_direction d
    ON q."Measure ID" = d."Measure ID"

WHERE q."Facility ID" = '490009'
  AND q."Score Numeric" IS NOT NULL

ORDER BY q."Quality Category", q."Measure Name";
"""

benchmark_status = pd.read_sql_query(query, conn)

benchmark_status

,Quality Category,Measure ID,Measure Name,UVA_Score,Virginia_Peer_Average,Performance Direction,Raw_Difference,Direction_Adjusted_Gap,Benchmark_Status
0,Infections,HAI_2_SIR,Catheter Associated Urinary Tract Infections (...,0.211,0.373796,LOWER,-0.163,0.163,Better Than VA Peer Average
1,Infections,HAI_1_SIR,Central Line Associated Bloodstream Infection ...,0.375,0.372023,LOWER,0.003,-0.003,Worse Than VA Peer Average
2,Infections,HAI_3_SIR,SSI - Colon Surgery,0.900,0.733477,LOWER,0.167,-0.167,Worse Than VA Peer Average
3,Mortality & Complications,PSI_04,Death rate among surgical inpatients with seri...,141.480,168.204000,LOWER,-26.724,26.724,Better Than VA Peer Average
4,Mortality & Complications,MORT_30_STK,Death rate for stroke patients,13.200,11.630769,LOWER,1.569,-1.569,Worse Than VA Peer Average
5,Mortality & Complications,PSI_08,In-hospital fall-associated fracture rate,0.370,0.262361,LOWER,0.108,-0.108,Worse Than VA Peer Average
6,Readmissions,READM_30_HF,Heart failure (HF) 30-Day Readmission Rate,19.900,21.246753,LOWER,-1.347,1.347,Better Than VA Peer Average
7,Readmissions,OP_35_ADM,Rate of inpatient admissions for patients rece...,11.000,10.864516,LOWER,0.135,-0.135,Worse Than VA Peer Average
8,Readmissions,OP_36,Ratio of unplanned hospital visits after hospi...,0.700,1.028125,LOWER,-0.328,0.328,Better Than VA Peer Average
9,Timely Care,OP_18d,Average (median) time patients spent in the em...,446.000,356.382979,LOWER,89.617,-89.617,Worse Than VA Peer Average


## Largest Standardized Peer Benchmark Gaps

Direction-adjusted z-scores identify the measures where UVA differs most from the Virginia peer distribution. The sign is adjusted using each measure's performance direction so positive values represent more favorable performance and negative values represent less favorable performance.

In [73]:
query = """
WITH MeasureStats AS (

    SELECT
        "Measure ID",
        AVG("Score Numeric") AS VA_Peer_Mean,
        AVG("Score Numeric" * "Score Numeric") AS Mean_Squared

    FROM hospital_quality

    WHERE "Score Numeric" IS NOT NULL
      AND "Facility ID" <> '490009'

    GROUP BY "Measure ID"
),

StatsWithSD AS (

    SELECT
        "Measure ID",
        VA_Peer_Mean,

        SQRT(
            MAX(
                Mean_Squared - (VA_Peer_Mean * VA_Peer_Mean),
                0
            )
        ) AS VA_Peer_StdDev

    FROM MeasureStats
),

UVAZScores AS (

    SELECT
        q."Quality Category",
        q."Measure ID",
        q."Measure Name",
        q."Score Numeric" AS UVA_Score,
        s.VA_Peer_Mean,
        (
            q."Score Numeric" - s.VA_Peer_Mean
        ) / NULLIF(s.VA_Peer_StdDev, 0) AS Raw_Z_Score,
        d."Performance Direction"

    FROM hospital_quality q

    JOIN StatsWithSD s
        ON q."Measure ID" = s."Measure ID"

    LEFT JOIN measure_direction d
        ON q."Measure ID" = d."Measure ID"

    WHERE q."Facility ID" = '490009'
      AND q."Score Numeric" IS NOT NULL
)

SELECT
    "Quality Category",
    "Measure ID",
    "Measure Name",
    UVA_Score,
    VA_Peer_Mean,

    ROUND(Raw_Z_Score, 3) AS Raw_Z_Score,

    ROUND(
        CASE
            WHEN "Performance Direction" = 'LOWER'
                THEN -Raw_Z_Score
            WHEN "Performance Direction" = 'HIGHER'
                THEN Raw_Z_Score
        END,
        3
    ) AS Direction_Adjusted_Z,

    "Performance Direction"

FROM UVAZScores

ORDER BY ABS(Raw_Z_Score) DESC;
"""

largest_gaps = pd.read_sql_query(query, conn)

largest_gaps

,Quality Category,Measure ID,Measure Name,UVA_Score,VA_Peer_Mean,Raw_Z_Score,Direction_Adjusted_Z,Performance Direction
0,Mortality & Complications,PSI_08,In-hospital fall-associated fracture rate,0.370,0.262361,3.256,-3.256,LOWER
1,Readmissions,OP_36,Ratio of unplanned hospital visits after hospi...,0.700,1.028125,-3.208,3.208,LOWER
2,Mortality & Complications,PSI_04,Death rate among surgical inpatients with seri...,141.480,168.204000,-1.398,1.398,LOWER
3,Timely Care,OP_18d,Average (median) time patients spent in the em...,446.000,356.382979,1.258,-1.258,LOWER
4,Timely Care,SAFE_USE_OF_OPIOIDS,Safe Use of Opioids - Concurrent Prescribing,10.000,14.120000,-1.195,1.195,LOWER
5,Readmissions,READM_30_HF,Heart failure (HF) 30-Day Readmission Rate,19.900,21.246753,-1.027,1.027,LOWER
6,Mortality & Complications,MORT_30_STK,Death rate for stroke patients,13.200,11.630769,0.826,-0.826,LOWER
7,Infections,HAI_2_SIR,Catheter Associated Urinary Tract Infections (...,0.211,0.373796,-0.442,0.442,LOWER
8,Infections,HAI_3_SIR,SSI - Colon Surgery,0.900,0.733477,0.275,-0.275,LOWER
9,Timely Care,OP_22,Left before being seen,2.000,1.670886,0.275,-0.275,LOWER


## Export Final Benchmark Dataset

The direction-adjusted benchmark status and standardized z-score outputs are combined into one analysis-ready table for Power BI and portfolio reporting.

In [74]:
benchmark_final = benchmark_status.merge(
    largest_gaps[
        [
            "Quality Category",
            "Measure ID",
            "Measure Name",
            "Raw_Z_Score",
            "Direction_Adjusted_Z"
        ]
    ],
    on=["Quality Category", "Measure ID", "Measure Name"],
    how="left"
)

benchmark_final.to_csv(
    "uva_complete_benchmark.csv",
    index=False
)

benchmark_final

,Quality Category,Measure ID,Measure Name,UVA_Score,Virginia_Peer_Average,Performance Direction,Raw_Difference,Direction_Adjusted_Gap,Benchmark_Status,Raw_Z_Score,Direction_Adjusted_Z
0,Infections,HAI_2_SIR,Catheter Associated Urinary Tract Infections (...,0.211,0.373796,LOWER,-0.163,0.163,Better Than VA Peer Average,-0.442,0.442
1,Infections,HAI_1_SIR,Central Line Associated Bloodstream Infection ...,0.375,0.372023,LOWER,0.003,-0.003,Worse Than VA Peer Average,0.009,-0.009
2,Infections,HAI_3_SIR,SSI - Colon Surgery,0.900,0.733477,LOWER,0.167,-0.167,Worse Than VA Peer Average,0.275,-0.275
3,Mortality & Complications,PSI_04,Death rate among surgical inpatients with seri...,141.480,168.204000,LOWER,-26.724,26.724,Better Than VA Peer Average,-1.398,1.398
4,Mortality & Complications,MORT_30_STK,Death rate for stroke patients,13.200,11.630769,LOWER,1.569,-1.569,Worse Than VA Peer Average,0.826,-0.826
5,Mortality & Complications,PSI_08,In-hospital fall-associated fracture rate,0.370,0.262361,LOWER,0.108,-0.108,Worse Than VA Peer Average,3.256,-3.256
6,Readmissions,READM_30_HF,Heart failure (HF) 30-Day Readmission Rate,19.900,21.246753,LOWER,-1.347,1.347,Better Than VA Peer Average,-1.027,1.027
7,Readmissions,OP_35_ADM,Rate of inpatient admissions for patients rece...,11.000,10.864516,LOWER,0.135,-0.135,Worse Than VA Peer Average,0.114,-0.114
8,Readmissions,OP_36,Ratio of unplanned hospital visits after hospi...,0.700,1.028125,LOWER,-0.328,0.328,Better Than VA Peer Average,-3.208,3.208
9,Timely Care,OP_18d,Average (median) time patients spent in the em...,446.000,356.382979,LOWER,89.617,-89.617,Worse Than VA Peer Average,1.258,-1.258


---

# Executive Summary

This project combines five public CMS hospital datasets to evaluate UVA Medical Center across 12 selected quality measures in four areas: timely care, readmissions/unplanned visits, healthcare-associated infections, and mortality/complications.

The datasets were cleaned and standardized in Python, then connected through the CMS Facility ID. National quality data was restricted to Virginia hospitals, and UVA Medical Center was separated from the peer benchmark population so its score would not influence the baseline used for comparison. Peer means, medians, standard deviations, raw differences, benchmark status, and direction-adjusted z-scores were calculated at the measure level.

SQL was then used to validate joins, reproduce peer benchmarks, classify UVA versus other Virginia hospitals, and perform standardized benchmark analysis. Because the selected measures use different units and scales, direction-adjusted z-scores provide the most consistent cross-measure comparison: positive values represent more favorable performance relative to Virginia peers and negative values represent less favorable performance.

The analysis shows a mixed performance profile rather than one uniform result. UVA performs more favorably than peers on some selected measures and less favorably on others, supporting a measure-level interpretation rather than a single combined hospital score. The final analysis-ready tables support the Power BI dashboard and provide a reproducible foundation for further hospital-quality analysis.

# Conclusion

This project demonstrates an end-to-end healthcare analytics workflow using public government data. Five CMS datasets were prepared and linked using Facility ID, quality measures were standardized and filtered to a Virginia comparison population, and UVA Medical Center was benchmarked against other reporting Virginia hospitals.

The project goes beyond simple raw-score comparisons by validating relational joins, excluding the focal hospital from its own peer benchmark, incorporating measure-specific performance direction, and standardizing differences with z-scores. These choices make comparisons across measures more transparent and reduce the risk of misleading interpretations when measures use different units or scales.

The resulting workflow demonstrates data cleaning, multi-table integration, quality assurance, statistical benchmarking, SQL analysis, and preparation of data for Power BI reporting. Rather than reducing hospital performance to one overall score, the analysis preserves measure-level differences so strengths and areas of weaker relative performance can be investigated individually.

## Tools Used

**Python • Pandas • NumPy • SQLite / SQL • Power BI • CMS Public Hospital Data**